# Analisis Dataset Bersama — Patokan Qwen3.5-0.8B (model terkecil)

Dijalankan **satu kali** untuk menghasilkan satu set data & parameter bersama ketiga notebook fine-tuning.

Tahapan:
1. Distribusi panjang token → rekomendasi `max_seq_length`
2. Estimasi kapasitas model vs ukuran data → rekomendasi jumlah sampel
3. Reduksi stratified → `train_reduced.jsonl`, `val_reduced.jsonl`, `training_config_recommended.json`

Output ditulis ke `Data/processed_shared/`. **Tidak butuh GPU.**


In [ ]:
%pip install -q transformers>=4.45.0
%pip install -q datasets>=3.0.0
%pip install -q numpy matplotlib


In [ ]:
import os, json, random
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ── Path: Colab (Drive) atau lokal Windows ─────────────────────────────────────
# Folder Drive HARUS punya struktur sama dengan project lokal: Data/, notebooks/, dll.
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Fine-Tune SLM for Medical Chatbot'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path(DRIVE_PROJECT_DIR)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    BASE_DIR = Path(os.getcwd()).parent
    if not (BASE_DIR / 'Data').exists():
        BASE_DIR = Path(os.getcwd())

assert (BASE_DIR / 'Data').exists(), f'Data/ tidak ditemukan di {BASE_DIR}'

DATA_DIR   = BASE_DIR / 'Data' / 'processed'
SHARED_DIR = BASE_DIR / 'Data' / 'processed_shared'
SHARED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = str(DATA_DIR / 'train.jsonl')
VAL_FILE   = str(DATA_DIR / 'val.jsonl')
OUT_DIR    = str(SHARED_DIR)

# Model patokan — terkecil dari 3 kandidat
BENCHMARK_MODEL_ID = 'Qwen/Qwen3.5-0.8B'
TOKENIZER_REF      = 'Qwen/Qwen3.5-0.8B'

STRATIFY_KEY      = 'source'
SEQ_PERCENTILE    = 95
TOKENIZE_PROC     = 1        # Windows: pakai 1 untuk hindari masalah multiprocessing

# Kuota Colab sangat terbatas (<10 jam GPU total untuk 3 model):
# mulai dari 20K sampel x 2 epoch untuk validasi pipeline (Qwen3.5-0.8B dulu).
# Setelah pipeline & eval_loss terbukti sehat dan kuota masih ada, naikkan bertahap.
TARGET_TRAIN_SAMPLES = 20_000
VAL_FRAC_OF_TRAIN    = 0.10    # ukuran val mengikuti proporsi train (bukan angka tetap)
MAX_SEQ_LEN          = None    # diisi dari analisis

LORA_R      = 16
LORA_ALPHA  = 32
LORA_TARGET = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
EPOCHS      = 2
SEED        = 42

print(f'Mode          : {"Colab (Drive)" if IN_COLAB else "Lokal"}')
print(f'Model patokan : {BENCHMARK_MODEL_ID}')
print(f'Base dir      : {BASE_DIR}')
print(f'Output shared : {OUT_DIR}')


## A1. Load dataset

In [ ]:
from datasets import load_dataset

train_raw = load_dataset('json', data_files=TRAIN_FILE, split='train')
val_raw   = load_dataset('json', data_files=VAL_FILE,   split='train')
print(f'Train : {len(train_raw):,}')
print(f'Val   : {len(val_raw):,}')
print(f'Kolom : {train_raw.column_names}')

HAS_SOURCE = STRATIFY_KEY in train_raw.column_names
if HAS_SOURCE:
    dist = Counter(train_raw[STRATIFY_KEY])
    print('\nDistribusi per sumber:')
    for k, v in dist.most_common():
        print(f'  {k:<22} {v:>7,}  ({v/len(train_raw)*100:.1f}%)')
else:
    print(f'\n[!] Kolom "{STRATIFY_KEY}" tidak ada -> reduksi acak global.')


## A2. Distribusi panjang token → max_seq_length

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(TOKENIZER_REF)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def _count_tokens_one(sample):
    try:
        text = tok.apply_chat_template(sample['messages'], tokenize=False,
                                        add_generation_prompt=False, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(sample['messages'], tokenize=False,
                                        add_generation_prompt=False)
    return len(tok(text, add_special_tokens=False)['input_ids'])

# Loop langsung di main process — hindari subprocess (datasets 5.x spawn worker bahkan num_proc=1)
print(f'Counting tokens for {len(train_raw):,} samples (main process) ...')
lengths = np.array([_count_tokens_one(s) for s in train_raw])
pct     = {q: int(np.percentile(lengths, q)) for q in [50, 90, 95, 99]}
round128 = lambda x: int(np.ceil(x / 128) * 128)
RECOMMENDED_MAX_SEQ = round128(pct[SEQ_PERCENTILE])

print(f'min/mean/max : {lengths.min()} / {lengths.mean():.0f} / {lengths.max()}')
for q in [50, 90, 95, 99]:
    print(f'  p{q:<2} : {pct[q]:,}')
cover = (lengths <= RECOMMENDED_MAX_SEQ).mean() * 100
print(f'\nRekomendasi max_seq_length (p{SEQ_PERCENTILE} -> kelipatan 128): {RECOMMENDED_MAX_SEQ}')
print(f'Sampel utuh (tanpa truncation): {cover:.2f}%')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(lengths, bins=60)
ax[0].axvline(RECOMMENDED_MAX_SEQ, color='r', ls='--', label=f'max_seq={RECOMMENDED_MAX_SEQ}')
ax[0].set_title('Distribusi panjang token'); ax[0].set_xlabel('token'); ax[0].legend()
xs = np.sort(lengths); ys = np.arange(1, len(xs)+1) / len(xs)
ax[1].plot(xs, ys)
ax[1].axvline(RECOMMENDED_MAX_SEQ, color='r', ls='--')
ax[1].axhline(SEQ_PERCENTILE/100, color='g', ls=':')
ax[1].set_title('CDF panjang token'); ax[1].set_xlabel('token'); ax[1].set_ylabel('proporsi')
plt.tight_layout()
plt.savefig(str(SHARED_DIR / 'token_length_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()


## A3. Kapasitas model vs ukuran data

In [ ]:
from transformers import AutoConfig

cfg  = AutoConfig.from_pretrained(TOKENIZER_REF)
tcfg = getattr(cfg, 'text_config', cfg)

H      = tcfg.hidden_size
I      = tcfg.intermediate_size
L      = tcfg.num_hidden_layers
hd     = getattr(tcfg, 'head_dim', H // tcfg.num_attention_heads)
q_out  = tcfg.num_attention_heads * hd
kv_out = getattr(tcfg, 'num_key_value_heads', tcfg.num_attention_heads) * hd

shapes = {'q_proj': (H, q_out), 'k_proj': (H, kv_out), 'v_proj': (H, kv_out),
          'o_proj': (q_out, H), 'gate_proj': (H, I), 'up_proj': (H, I), 'down_proj': (I, H)}
per_layer  = sum(LORA_R * (i + o) for n, (i, o) in shapes.items() if n in LORA_TARGET)
lora_params = per_layer * L

print(f'hidden={H}  inter={I}  layers={L}')
print(f'Trainable LoRA params (r={LORA_R}) : {lora_params/1e6:.2f} M')

total_tokens_full = int(lengths.sum())
print(f'\nTotal token train ({len(lengths):,} sampel) : {total_tokens_full/1e6:.1f} M token')
print('\nPanduan ukuran data (LoRA, model sub-2B):')
for n in [10_000, 15_000, 20_000, 25_000, 30_000]:
    est_tok = int(lengths.mean() * n)
    note = '(kecil, risiko overfit)' if n < 15_000 else ('(aman)' if n <= 30_000 else '')
    print(f'  {n:>6,} sampel  ~{est_tok/1e6:5.1f}M token x {EPOCHS} epoch  {note}')
RECOMMENDED_N = min(20_000, len(train_raw))
print(f'\nRekomendasi awal TARGET_TRAIN_SAMPLES : {RECOMMENDED_N:,}')


## A4. Set parameter final & simpan config

In [ ]:
MAX_SEQ_LEN          = MAX_SEQ_LEN          or RECOMMENDED_MAX_SEQ
TARGET_TRAIN_SAMPLES = TARGET_TRAIN_SAMPLES or RECOMMENDED_N

rec = {
    'model': BENCHMARK_MODEL_ID,
    'max_seq_length': MAX_SEQ_LEN,
    'max_seq_percentile': SEQ_PERCENTILE,
    'token_percentiles': pct,
    'token_coverage_pct': round(float((lengths <= MAX_SEQ_LEN).mean() * 100), 2),
    'train_samples_full': len(train_raw),
    'target_train_samples': TARGET_TRAIN_SAMPLES,
    'lora_trainable_params_million': round(lora_params / 1e6, 2),
    'epochs': EPOCHS,
    'seed': SEED,
}
with open(str(SHARED_DIR / 'training_config_recommended.json'), 'w') as f:
    json.dump(rec, f, indent=2)
print(json.dumps(rec, indent=2))


## A5. Reduksi stratified

In [ ]:
def stratified_reduce(ds, target_n, key, seed=SEED):
    target_n = min(target_n, len(ds))
    rng = random.Random(seed)
    if key not in ds.column_names:
        idx = list(range(len(ds))); rng.shuffle(idx)
        return ds.select(sorted(idx[:target_n]))
    buckets = defaultdict(list)
    for i, k in enumerate(ds[key]):
        buckets[k].append(i)
    total = len(ds); chosen = []
    for k, idxs in buckets.items():
        rng.shuffle(idxs)
        chosen += idxs[:max(1, round(target_n * len(idxs) / total))]
    rng.shuffle(chosen)
    return ds.select(sorted(chosen[:target_n]))

VAL_TARGET = round(TARGET_TRAIN_SAMPLES * VAL_FRAC_OF_TRAIN)

train_red = stratified_reduce(train_raw, TARGET_TRAIN_SAMPLES, STRATIFY_KEY)
val_red   = stratified_reduce(val_raw,   min(VAL_TARGET, len(val_raw)), STRATIFY_KEY)
print(f'Train : {len(train_raw):,} -> {len(train_red):,}')
print(f'Val   : {len(val_raw):,} -> {len(val_red):,}')

if HAS_SOURCE:
    before = Counter(train_raw[STRATIFY_KEY]); after = Counter(train_red[STRATIFY_KEY])
    print('\nSumber              before%  after%')
    for k in sorted(before):
        print(f'  {k:<20} {before[k]/len(train_raw)*100:5.1f}   {after[k]/len(train_red)*100:5.1f}')

TRAIN_RED_FILE = str(SHARED_DIR / 'train_reduced.jsonl')
VAL_RED_FILE   = str(SHARED_DIR / 'val_reduced.jsonl')

train_red.select_columns(['messages']).to_json(TRAIN_RED_FILE, lines=True, force_ascii=False)
val_red.select_columns(['messages']).to_json(VAL_RED_FILE,   lines=True, force_ascii=False)
print(f'\nTrain reduced -> {TRAIN_RED_FILE}')
print(f'Val   reduced -> {VAL_RED_FILE}')
print(f'Config        -> {SHARED_DIR / "training_config_recommended.json"}')
